# Vulnerability Classifier Teaching Notebook

This notebook is a compact, single-file version of the GraphCodeBERT classifier workflow used in the training notebooks. It loads the same Hugging Face dataset, trains in two phases, then evaluates with threshold sweeps and confusion matrices derived from real predictions.

For reference, the working multi-stage training flow lives in [exploit_detection.ipynb](/workspaces/vuln-classification/exploit_detection.ipynb).

In [ ]:
!pip install transformers datasets scikit-learn accelerate huggingface_hub evaluate sentencepiece matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 19.1 MB/s  0:00:00m0:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 660.6/660.6 kB 15.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 42.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 41.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 18.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 41.5 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 29.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 37.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 57.2 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 56.3 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 801.1/801.1 kB 33.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.2/35.2 MB 32.9 MB/s  0:00:016m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

In [4]:
!pip install matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 54.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 55.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 49.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 51.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [matplotlib]7 [matplotlib]

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [3]:
import json
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import Dataset, load_dataset
from sklearn.metrics import (
    f1_score, precision_score, recall_score, confusion_matrix,
    multilabel_confusion_matrix, classification_report,
)
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    DataCollatorWithPadding, Trainer, TrainingArguments,
)

sns.set(style='whitegrid')
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DATASET_ID = 'ayshajavd/code-security-vulnerability-dataset'
MODEL_NAME = 'microsoft/graphcodebert-base'
MAX_LENGTH = 512
PHASE1_DIR = './phase1_checkpoint'
PHASE2_DIR = './phase2_checkpoint'
TEST_THRESHOLD = 0.3

TARGET_CWES = [
    'safe', 'CWE-20', 'CWE-22', 'CWE-78', 'CWE-79', 'CWE-89', 'CWE-94',
    'CWE-119', 'CWE-125', 'CWE-190', 'CWE-200', 'CWE-264', 'CWE-269',
    'CWE-276', 'CWE-284', 'CWE-287', 'CWE-310', 'CWE-327', 'CWE-330',
    'CWE-352', 'CWE-362', 'CWE-399', 'CWE-401', 'CWE-416', 'CWE-434',
    'CWE-476', 'CWE-502', 'CWE-601', 'CWE-787', 'CWE-798', 'CWE-918',
]
NUM_LABELS = len(TARGET_CWES)
label2id = {label: idx for idx, label in enumerate(TARGET_CWES)}
id2label = {idx: label for label, idx in label2id.items()}

CWE_NAMES = {
    'safe': 'Safe Code',
    'CWE-20': 'Improper Input Validation',
    'CWE-22': 'Path Traversal',
    'CWE-78': 'OS Command Injection',
    'CWE-79': 'Cross-Site Scripting',
    'CWE-89': 'SQL Injection',
    'CWE-94': 'Code Injection',
    'CWE-119': 'Buffer Overflow',
    'CWE-125': 'Out-of-bounds Read',
    'CWE-190': 'Integer Overflow',
    'CWE-200': 'Information Exposure',
    'CWE-264': 'Permissions/Privileges',
    'CWE-269': 'Improper Privilege Mgmt',
    'CWE-276': 'Incorrect Permissions',
    'CWE-284': 'Access Control',
    'CWE-287': 'Authentication',
    'CWE-310': 'Cryptographic Issues',
    'CWE-327': 'Broken Crypto Algorithm',
    'CWE-330': 'Insufficient Randomness',
    'CWE-352': 'CSRF',
    'CWE-362': 'Race Condition',
    'CWE-399': 'Resource Management',
    'CWE-401': 'Memory Leak',
    'CWE-416': 'Use After Free',
    'CWE-434': 'File Upload',
    'CWE-476': 'NULL Pointer Deref',
    'CWE-502': 'Deserialization',
    'CWE-601': 'Open Redirect',
    'CWE-787': 'Out-of-bounds Write',
    'CWE-798': 'Hardcoded Credentials',
    'CWE-918': 'SSRF',
}


def detect_columns(split):
    code_candidates = ['code', 'func', 'text', 'source']
    label_candidates = ['labels', 'label', 'targets']
    code_col = next((name for name in code_candidates if name in split.column_names), None)
    label_col = next((name for name in label_candidates if name in split.column_names), None)
    if code_col is None:
        raise RuntimeError(f'Could not find a code column in: {split.column_names}')
    if label_col is None:
        raise RuntimeError(f'Could not find a label column in: {split.column_names}')
    return code_col, label_col


def coerce_multilabel_matrix(values, label_col):
    first = values[0]
    if isinstance(first, (list, tuple, np.ndarray)):
        matrix = np.asarray(values, dtype=np.int64)
        if matrix.ndim != 2 or matrix.shape[1] != NUM_LABELS:
            raise RuntimeError(f'Unexpected label matrix shape {matrix.shape} in column {label_col}')
        return matrix
    if isinstance(first, str):
        one_hot = np.zeros((len(values), NUM_LABELS), dtype=np.int64)
        for row_index, label in enumerate(values):
            if label not in label2id:
                raise RuntimeError(f'Unknown label {label}')
            one_hot[row_index, label2id[label]] = 1
        return one_hot
    raise RuntimeError(f'Unsupported label format in {label_col}: {type(first)}')


def build_compute_metrics(threshold):
    def compute_metrics(eval_pred):
        logits = eval_pred.predictions
        labels = eval_pred.label_ids
        probs = 1.0 / (1.0 + np.exp(-logits))
        preds = (probs > threshold).astype(int)
        metrics = {
            'macro_f1': f1_score(labels, preds, average='macro', zero_division=0),
            'micro_f1': f1_score(labels, preds, average='micro', zero_division=0),
            'weighted_f1': f1_score(labels, preds, average='weighted', zero_division=0),
            'macro_precision': precision_score(labels, preds, average='macro', zero_division=0),
            'macro_recall': recall_score(labels, preds, average='macro', zero_division=0),
        }
        per_class_f1 = f1_score(labels, preds, average=None, zero_division=0)
        per_class_precision = precision_score(labels, preds, average=None, zero_division=0)
        per_class_recall = recall_score(labels, preds, average=None, zero_division=0)
        for idx, cwe in enumerate(TARGET_CWES):
            metrics[f'f1_{cwe}'] = float(per_class_f1[idx])
            metrics[f'precision_{cwe}'] = float(per_class_precision[idx])
            metrics[f'recall_{cwe}'] = float(per_class_recall[idx])
        return metrics
    return compute_metrics


class AsymmetricLoss(nn.Module):
    def __init__(self, gamma_neg=4, gamma_pos=0, clip=0.05, eps=1e-8):
        super().__init__()
        self.gamma_neg = gamma_neg
        self.gamma_pos = gamma_pos
        self.clip = clip
        self.eps = eps

    def forward(self, logits, targets):
        xs_pos = torch.sigmoid(logits)
        xs_neg = 1.0 - xs_pos
        if self.clip is not None and self.clip > 0:
            xs_neg = (xs_neg + self.clip).clamp(max=1.0)
        loss_pos = targets * torch.log(xs_pos.clamp(min=self.eps))
        loss_neg = (1.0 - targets) * torch.log(xs_neg.clamp(min=self.eps))
        loss = loss_pos + loss_neg
        if self.gamma_neg > 0 or self.gamma_pos > 0:
            pt0 = xs_pos * targets
            pt1 = xs_neg * (1.0 - targets)
            pt = pt0 + pt1
            one_sided_gamma = self.gamma_pos * targets + self.gamma_neg * (1.0 - targets)
            one_sided_w = torch.pow(1.0 - pt, one_sided_gamma)
            loss = loss * one_sided_w
        return -loss.sum() / logits.shape[0]


class ASLTrainer(Trainer):
    def __init__(self, *args, loss_fn=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.loss_fn = loss_fn or AsymmetricLoss()

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop('labels').float()
        outputs = model(**inputs)
        loss = self.loss_fn(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

ModuleNotFoundError: No module named 'matplotlib'

## Load the dataset and inspect imbalance

This uses the same Hugging Face dataset as the source notebooks. The class imbalance plot is computed from the actual training split at runtime.

In [ ]:
ds = load_dataset(DATASET_ID)
print(ds)

if 'train' not in ds or 'validation' not in ds or 'test' not in ds:
    raise RuntimeError(f'Expected train/validation/test splits, got: {list(ds.keys())}')

train_raw = ds['train']
val_raw = ds['validation']
test_raw = ds['test']
code_col, label_col = detect_columns(train_raw)

train_labels = coerce_multilabel_matrix(train_raw[label_col], label_col)
val_labels = coerce_multilabel_matrix(val_raw[label_col], label_col)
test_labels = coerce_multilabel_matrix(test_raw[label_col], label_col)

class_supports = train_labels.sum(axis=0).astype(int)
class_support_df = pd.DataFrame({
    'label': TARGET_CWES,
    'support': class_supports,
    'name': [CWE_NAMES.get(label, label) for label in TARGET_CWES],
})

print(f'Code column: {code_col}')
print(f'Label column: {label_col}')
print(class_support_df.sort_values('support', ascending=False).head(12).to_string(index=False))

plt.figure(figsize=(12, 5))
sns.barplot(data=class_support_df.sort_values('support', ascending=False), x='label', y='support', palette='viridis')
plt.xticks(rotation=75, ha='right')
plt.title('Training Class Imbalance')
plt.ylabel('Support')
plt.xlabel('CWE')
plt.tight_layout()
plt.show()

## Training and evaluation

The classifier follows the same two-phase GraphCodeBERT sequence as the source notebooks: phase 1 freezes the bottom eight layers, phase 2 fine-tunes the full model, and the final test pass generates thresholds and confusion matrices from real predictions.

In [ ]:
os.makedirs(PHASE1_DIR, exist_ok=True)
os.makedirs(PHASE2_DIR, exist_ok=True)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=id2label,
    label2id=label2id,
    problem_type='multi_label_classification',
)

with torch.no_grad():
    bias_param = None
    if hasattr(model, 'classifier'):
        for name, param in model.classifier.named_parameters():
            if name.endswith('bias') and param.shape[0] == NUM_LABELS:
                bias_param = param
                break
    if bias_param is not None:
        priors = np.clip(class_supports / len(train_labels), 1e-5, 1 - 1e-5)
        bias_values = -np.log((1.0 - priors) / priors)
        bias_param.copy_(torch.tensor(bias_values, dtype=bias_param.dtype))
        print(f'Bias initialized from train split: [{bias_values.min():.2f}, {bias_values.max():.2f}]')


def prepare_split(split):
    def encode(batch):
        tokenized = tokenizer(batch[code_col], truncation=True, max_length=MAX_LENGTH)
        tokenized['labels'] = coerce_multilabel_matrix(batch[label_col], label_col).tolist()
        return tokenized

    encoded = split.map(encode, batched=True, batch_size=64)
    drop_columns = [column for column in encoded.column_names if column not in ['input_ids', 'attention_mask', 'labels']]
    encoded = encoded.remove_columns(drop_columns)
    encoded.set_format('torch')
    return encoded


train_ds = prepare_split(train_raw)
val_ds = prepare_split(val_raw)
test_ds = prepare_split(test_raw)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
print(f'Train rows: {len(train_ds):,} | Val rows: {len(val_ds):,} | Test rows: {len(test_ds):,}')

# Phase 1: freeze embeddings + bottom 8 layers.
for param in model.roberta.embeddings.parameters():
    param.requires_grad = False
for layer in model.roberta.encoder.layer[:8]:
    for param in layer.parameters():
        param.requires_grad = False

phase1_args = TrainingArguments(
    output_dir=PHASE1_DIR,
    num_train_epochs=4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=2,
    learning_rate=1e-4,
    lr_scheduler_type='cosine',
    warmup_ratio=0.06,
    weight_decay=0.01,
    max_grad_norm=1.0,
    fp16=torch.cuda.is_available(),
    evaluation_strategy='epoch',
    save_strategy='epoch',
    logging_strategy='steps',
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model='eval_macro_f1',
    greater_is_better=True,
    save_total_limit=2,
    seed=SEED,
    dataloader_num_workers=2,
    report_to='none',
    label_names=['labels'],
)

phase1_trainer = ASLTrainer(
    model=model,
    args=phase1_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=data_collator,
    compute_metrics=build_compute_metrics(TEST_THRESHOLD),
    loss_fn=AsymmetricLoss(gamma_neg=4, gamma_pos=0, clip=0.05),
)

print('Starting phase 1 training...')
phase1_trainer.train()
phase1_metrics = phase1_trainer.evaluate()
print('Phase 1 metrics:', phase1_metrics)
phase1_trainer.model.save_pretrained(PHASE1_DIR)
tokenizer.save_pretrained(PHASE1_DIR)

# Phase 2: reload checkpoint and fine-tune all layers.
model = AutoModelForSequenceClassification.from_pretrained(
    PHASE1_DIR,
    num_labels=NUM_LABELS,
    id2label=id2label,
    label2id=label2id,
    problem_type='multi_label_classification',
)
for param in model.parameters():
    param.requires_grad = True

phase2_args = TrainingArguments(
    output_dir=PHASE2_DIR,
    num_train_epochs=9,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=2,
    learning_rate=2e-5,
    lr_scheduler_type='cosine',
    warmup_ratio=0.06,
    weight_decay=0.01,
    max_grad_norm=1.0,
    fp16=torch.cuda.is_available(),
    evaluation_strategy='epoch',
    save_strategy='epoch',
    logging_strategy='steps',
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model='eval_macro_f1',
    greater_is_better=True,
    save_total_limit=3,
    seed=SEED,
    dataloader_num_workers=2,
    report_to='none',
    label_names=['labels'],
)

phase2_trainer = ASLTrainer(
    model=model,
    args=phase2_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=data_collator,
    compute_metrics=build_compute_metrics(TEST_THRESHOLD),
    loss_fn=AsymmetricLoss(gamma_neg=4, gamma_pos=0, clip=0.05),
)

print('Starting phase 2 training...')
phase2_trainer.train()
phase2_metrics = phase2_trainer.evaluate()
print('Phase 2 metrics:', phase2_metrics)
phase2_trainer.model.save_pretrained(PHASE2_DIR)
tokenizer.save_pretrained(PHASE2_DIR)

# Final test evaluation.
test_outputs = phase2_trainer.predict(test_ds)
test_logits = test_outputs.predictions
test_labels = test_outputs.label_ids
test_probs = 1.0 / (1.0 + np.exp(-test_logits))

threshold_rows = []
for threshold in [0.2, 0.3, 0.4, 0.5]:
    test_preds_at_t = (test_probs > threshold).astype(int)
    threshold_rows.append({
        'threshold': threshold,
        'macro_f1': f1_score(test_labels, test_preds_at_t, average='macro', zero_division=0),
        'micro_f1': f1_score(test_labels, test_preds_at_t, average='micro', zero_division=0),
        'weighted_f1': f1_score(test_labels, test_preds_at_t, average='weighted', zero_division=0),
        'macro_precision': precision_score(test_labels, test_preds_at_t, average='macro', zero_division=0),
        'macro_recall': recall_score(test_labels, test_preds_at_t, average='macro', zero_division=0),
    })
threshold_df = pd.DataFrame(threshold_rows)
print('Threshold comparison:')
display(threshold_df)

chosen_threshold = TEST_THRESHOLD
test_preds = (test_probs > chosen_threshold).astype(int)
per_class_df = pd.DataFrame({
    'label': TARGET_CWES,
    'name': [CWE_NAMES.get(label, label) for label in TARGET_CWES],
    'support': test_labels.sum(axis=0).astype(int),
    'precision': precision_score(test_labels, test_preds, average=None, zero_division=0),
    'recall': recall_score(test_labels, test_preds, average=None, zero_division=0),
    'f1': f1_score(test_labels, test_preds, average=None, zero_division=0),
})
print('Per-class performance at threshold=0.3:')
display(per_class_df.sort_values(['f1', 'support'], ascending=[False, False]))

vuln_df = per_class_df[per_class_df['label'] != 'safe'].copy()
vuln_df = vuln_df[vuln_df['support'] >= 10].sort_values(['f1', 'support'], ascending=[False, False]).reset_index(drop=True)
top5 = vuln_df.head(5).copy()
print('Top-5 vulnerability classes by F1 with support >= 10:')
display(top5)

# Exact one-vs-rest confusion matrices for the top five vulnerability classes.
label_indices = [label2id[label] for label in top5['label'].tolist()]
label_confusions = multilabel_confusion_matrix(test_labels, test_preds)

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()
for ax, label_index in zip(axes, label_indices):
    cm = label_confusions[label_index]
    sns.heatmap(
        cm,
        annot=True,
        fmt='d',
        cmap='Blues',
        cbar=False,
        xticklabels=['Pred 0', 'Pred 1'],
        yticklabels=['True 0', 'True 1'],
        ax=ax,
    )
    label = TARGET_CWES[label_index]
    ax.set_title(f"{label} — {CWE_NAMES.get(label, label)}")
    ax.set_xlabel('')
    ax.set_ylabel('')

for ax in axes[len(label_indices):]:
    ax.axis('off')

plt.suptitle('Top-5 Vulnerability Class Confusion Matrices (threshold=0.3)', y=1.02)
plt.tight_layout()
plt.show()

print('Classification report (threshold=0.3):')
print(classification_report(test_labels, test_preds, target_names=TARGET_CWES, zero_division=0, digits=4))